Silver Layer - Reference Data Transformation
📌 Purpose

This notebook transforms bronze reference tables into clean silver tables.

What this notebook does:
Reads bronze reference tables
Cleans string values
Adds useful business columns
Creates standardized silver tables
Saves refined data for analytics

In [0]:
import yaml
from pyspark.sql import functions as F

In [0]:
import yaml
from pyspark.sql import functions as F

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
silver_schema = "refined"

bronze_db = f"{catalog}.{raw_schema}"
silver_table = f"{catalog}.{silver_schema}.silver_reference_all"

In [0]:
weather_df = spark.table("vattenfall_dev.raw.ref_weather_alert_reference")
region_df = spark.table("vattenfall_dev.raw.ref_region_reference")

print("Loaded bronze tables")

We remove extra spaces from all string columns to clean the data.

In [0]:
from pyspark.sql import functions as F

# Trim strings (safe)
for col_name, dtype in weather_df.dtypes:
    if dtype == "string":
        weather_df = weather_df.withColumn(col_name, F.trim(F.col(col_name)))

for col_name, dtype in region_df.dtypes:
    if dtype == "string":
        region_df = region_df.withColumn(col_name, F.trim(F.col(col_name)))

### We are creating a refined (silver layer) weather dataset with new useful columns for analysis.

What we do here:

alert_key: we copy weather_alert_level so it can be used as a standard key
is_severe: we create a flag (True/False)
If priority_rank = 3, it means the alert is severe → True
Otherwise → False

This helps in quickly identifying critical weather alerts.

In [0]:
silver_weather = weather_df \
    .withColumn("alert_key", F.col("weather_alert_level")) \
    .withColumn("is_severe", 
                F.when(F.col("priority_rank") == 3, True).otherwise(False))

## Create Silver Region Dataset

### We are preparing region data by creating keys that help in joins and grouping.

What we do here:

region_key: simple copy of region column (used as identifier)
country_region_key: combines country + region (like "DE_North")

This helps in uniquely identifying regions across countries.

In [0]:
silver_region = region_df \
    .withColumn("region_key", F.col("region")) \
    .withColumn("country_region_key", 
                F.concat(F.col("country_code"), F.lit("_"), F.col("region")))

###We are saving processed datasets into database tables (silver layer).

What this means:

We store cleaned + transformed data permanently
overwrite means old data will be replaced
These tables are now ready for reporting and analytics

In [0]:
silver_weather.write.mode("overwrite").saveAsTable(
    "vattenfall_dev.refined.silver_weather_alert"
)

silver_region.write.mode("overwrite").saveAsTable(
    "vattenfall_dev.refined.silver_region"
)

print("Silver tables created successfully")

In [0]:
display(spark.table("vattenfall_dev.refined.silver_weather_alert"))
display(spark.table("vattenfall_dev.refined.silver_region"))